In [ ]:
# ---------------------------------------------------------------------------
# Path setup -- fixes imports and data paths when notebook runs from subdir
# ---------------------------------------------------------------------------
from pathlib import Path
import sys, os

REPO_ROOT = Path().resolve().parent.parent  # notebooks/{subdir}/ -> root
os.chdir(REPO_ROOT)                          # all relative paths (cache/, data/, results/, splits/) resolve from root
sys.path.insert(0, str(REPO_ROOT / 'src'))   # find rise, crise, synth_gen modules

# CRISE-ID — Temperature (τ) Hyperparameter Tuning

**Research question:** Is τ = 0.1 the optimal softmax temperature for CRISE-ID?

The softmax weighting function is:

```
w = softmax(sims / τ)[true_id]
```

A lower τ sharpens the distribution (more one-hot); a higher τ flattens it toward uniform.
We ablate τ ∈ {0.05, 0.1, 0.2, 0.5} and evaluate each using insertion/deletion AUC
(identification margin) on a fixed random sample of real probes.

**τ = 0.1 maps are already cached — only the remaining three tau values are recomputed.**

Output: `results/tau_ablation/`


In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

TAU_VALUES   = [0.05, 0.1, 0.2, 0.5]   # ablation grid
TAU_DEFAULT  = 0.1                       # already cached; skip recomputation

N_SAMPLE     = 300    # probes per tau — enough for stable AUC estimates
SAMPLE_SEED  = 42
EVAL_STEPS   = 50
N_MASKS      = 1000
S            = 8
P1           = 0.5
MASTER_SEED  = 123

GALLERY_EMB  = "cache/G.npy"
GALLERY_IDS  = "cache/gallery_ids.npy"
PROBE_META   = "cache/probe_meta.json"
SPLIT_PATH   = "splits/lfw_1N_split.json"

CRISE_MAP_DIR = "results/crise_maps"        # existing τ=0.1 maps live here
TAU_ABL_DIR   = "results/tau_ablation"      # new tau maps go here

# Pre-computed τ=0.1 eval CSV (skip recomputation)
TAU01_EVAL_CSV = "results/crise_maps/eval_margin_auc/eval_margin_auc_steps50_black.csv"

# RISE baseline for comparison
RISE_EVAL_CSV  = "results/rise_baseline/eval_margin_auc_multi/eval_margin_auc_multi_steps50_black.csv"

import os
os.makedirs(TAU_ABL_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm
from insightface.app import FaceAnalysis

from rise import run_rise, build_aligned_chip_112, get_embedding_from_chip
from crise import run_crise, softmax_weight

print("Imports OK")

In [ ]:
# ---------------------------------------------------------------------------
# InsightFace setup
# ---------------------------------------------------------------------------

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
app.prepare(ctx_id=0, det_size=(640, 640))
rec = app.models["recognition"]
print("Models loaded:", list(app.models.keys()))

In [ ]:
# ---------------------------------------------------------------------------
# Load gallery + sample probes
# Use the same fixed sample for all tau values so comparisons are paired.
# ---------------------------------------------------------------------------

G            = np.load(GALLERY_EMB).astype(np.float32)
gallery_ids  = np.load(GALLERY_IDS, allow_pickle=True).tolist()
id_to_index  = {name: i for i, name in enumerate(gallery_ids)}

with open(PROBE_META) as f:
    probe_meta = json.load(f)

# Filter to valid probes (face detected, gallery id known)
BAD_GALLERY = {"Emile_Lahoud", "John_Paul_II"}
valid_probes = [
    p for p in probe_meta
    if p.get("ok", False)
    and p["true_id"] in id_to_index
    and p["true_id"] not in BAD_GALLERY
]
print(f"Valid probes: {len(valid_probes)}")

# Fixed random sample — same across all tau runs
rng    = np.random.default_rng(SAMPLE_SEED)
idx    = rng.choice(len(valid_probes), size=min(N_SAMPLE, len(valid_probes)), replace=False)
sample = [valid_probes[i] for i in idx]

print(f"Sample size : {len(sample)}")
print(f"Sample seed : {SAMPLE_SEED}")

In [ ]:
# ---------------------------------------------------------------------------
# Insertion / deletion AUC helpers (identification margin)
#
# margin(x) = sim(true_id) - max(sim(all_impostors))
# Deletion: mask top-k salient pixels with black, measure margin drop
# Insertion: start from black, reveal pixels in saliency order, measure rise
# ---------------------------------------------------------------------------

_CHIP_H, _CHIP_W = 112, 112
_N_PIX = _CHIP_H * _CHIP_W

def margin_score(chip_bgr, true_idx):
    try:
        emb  = get_embedding_from_chip(chip_bgr, rec)
        sims = G @ emb
        return float(sims[true_idx] - np.delete(sims, true_idx).max())
    except Exception:
        return float("nan")

def compute_auc(chip_bgr, sal, true_idx, steps=EVAL_STEPS):
    '''Return (deletion_auc, insertion_auc, margin_clean).'''
    order    = np.argsort(sal.ravel())[::-1]   # descending saliency
    budgets  = np.linspace(0, 1, steps + 1)
    del_margins = []
    ins_margins = []

    for budget in budgets:
        k  = int(_N_PIX * budget)
        ys, xs = np.unravel_index(order[:k] if k > 0 else np.array([], dtype=np.intp), (_CHIP_H, _CHIP_W))

        # Deletion: black-mask top-k
        del_chip = chip_bgr.copy().astype(np.uint8)
        if k > 0:
            del_chip[ys, xs] = 0
        del_margins.append(margin_score(del_chip, true_idx))

        # Insertion: start black, reveal top-k
        ins_chip = np.zeros_like(chip_bgr, dtype=np.uint8)
        if k > 0:
            ins_chip[ys, xs] = chip_bgr[ys, xs]
        ins_margins.append(margin_score(ins_chip, true_idx))

    del_arr = np.array(del_margins, dtype=np.float32)
    ins_arr = np.array(ins_margins, dtype=np.float32)

    # Replace NaN with linear interpolation
    for arr in [del_arr, ins_arr]:
        nans = np.isnan(arr)
        if nans.any() and (~nans).any():
            arr[nans] = np.interp(np.where(nans)[0],
                                  np.where(~nans)[0], arr[~nans])

    del_auc = float(np.trapz(del_arr, budgets))
    ins_auc = float(np.trapz(ins_arr, budgets))
    return del_auc, ins_auc, del_margins[0]   # margin_clean = budget=0 deletion

print("Evaluation helpers ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Run CRISE on the fixed sample for tau values OTHER than 0.1
# τ = 0.1 maps are already cached — we load their eval CSV directly.
# Each tau gets its own subdirectory under TAU_ABL_DIR.
# Idempotent: skips probes whose saliency map already exists.
# ---------------------------------------------------------------------------

for tau in TAU_VALUES:
    if tau == TAU_DEFAULT:
        print(f"tau={tau}: skipping (already cached, will load from eval CSV)")
        continue

    tau_str  = f"{tau:.3g}".replace(".", "p")
    tau_dir  = os.path.join(TAU_ABL_DIR, f"tau_{tau_str}")
    os.makedirs(tau_dir, exist_ok=True)

    print(f"\ntau={tau} → {tau_dir}")
    n_done = 0; n_skip = 0; n_fail = 0

    for i, probe in enumerate(tqdm(sample, desc=f"CRISE tau={tau}")):
        probe_path = probe["img_path"]
        true_id    = probe["true_id"]
        run_seed   = MASTER_SEED * 10_000 + i * 100

        # Check if already cached
        pid      = true_id
        img_stem = os.path.splitext(os.path.basename(probe_path))[0]
        prefix   = f"crise_tau{tau_str}"
        expected = os.path.join(
            tau_dir,
            f"{prefix}_{pid}_{img_stem}_N{N_MASKS}_s{S}_p{P1}_seed{run_seed}_saliency_norm.npy"
        )
        if os.path.exists(expected):
            n_skip += 1
            continue

        try:
            run_crise(
                true_id     = true_id,
                probe_path  = probe_path,
                G           = G,
                id_to_index = id_to_index,
                rec         = rec,
                app         = app,
                run_seed    = run_seed,
                out_dir     = tau_dir,
                tau         = tau,
                N           = N_MASKS,
                s           = S,
                p1          = P1,
            )
            n_done += 1
        except Exception as e:
            n_fail += 1

    print(f"  done={n_done}  skipped={n_skip}  failed={n_fail}")

In [ ]:
# ---------------------------------------------------------------------------
# Load τ=0.1 AUC from existing eval CSV (subsample to match our fixed sample).
# For other tau values: locate the cached saliency map, compute AUC on the fly.
# ---------------------------------------------------------------------------

# Build set of probe_paths in our fixed sample for fast lookup
sample_paths = {p["img_path"] for p in sample}
sample_id    = {p["img_path"]: p["true_id"] for p in sample}

auc_results = {}   # tau -> list of {del_auc, ins_auc, margin_clean}

# ── τ = 0.1: load from pre-computed eval CSV ────────────────────────────────
tau01_df = pd.read_csv(TAU01_EVAL_CSV)
tau01_sub = tau01_df[tau01_df["probe_path"].isin(sample_paths)].copy()
print(f"tau=0.1: loaded {len(tau01_sub)} pre-computed AUC rows")

auc_results[0.1] = [
    {"del_auc": row["del_auc_margin"],
     "ins_auc": row["ins_auc_margin"],
     "margin_clean": row["margin_clean"]}
    for _, row in tau01_sub.iterrows()
]

# ── Other tau values: compute AUC from cached maps ──────────────────────────
for tau in TAU_VALUES:
    if tau == TAU_DEFAULT:
        continue

    tau_str = f"{tau:.3g}".replace(".", "p")
    tau_dir = os.path.join(TAU_ABL_DIR, f"tau_{tau_str}")
    records = []

    for probe in tqdm(sample, desc=f"AUC tau={tau}"):
        probe_path = probe["img_path"]
        true_id    = probe["true_id"]
        true_idx   = id_to_index[true_id]

        # Find the saliency map for this probe + tau
        prefix  = f"crise_tau{tau_str}"
        img_stem = os.path.splitext(os.path.basename(probe_path))[0]
        # glob-style search in case seed encoding differs
        candidates = [
            f for f in os.listdir(tau_dir)
            if f.startswith(f"{prefix}_{true_id}_{img_stem}_")
            and f.endswith("_saliency_norm.npy")
        ]
        if not candidates:
            continue

        sal_path = os.path.join(tau_dir, candidates[0])
        sal = np.load(sal_path).astype(np.float32)

        img = __import__("cv2").imread(probe_path)
        if img is None:
            continue
        try:
            chip = build_aligned_chip_112(img, app)
        except Exception:
            continue

        del_auc, ins_auc, m_clean = compute_auc(chip, sal, true_idx)
        records.append({"del_auc": del_auc, "ins_auc": ins_auc,
                        "margin_clean": m_clean})

    auc_results[tau] = records
    print(f"  tau={tau}: {len(records)} probes evaluated")

print("\nAUC computation complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Sensitivity table: mean Deletion/Insertion AUC per tau
# Also load RISE baseline for comparison.
# ---------------------------------------------------------------------------

rows = []
for tau in TAU_VALUES:
    recs = auc_results.get(tau, [])
    if not recs:
        continue
    del_aucs = [r["del_auc"] for r in recs if not np.isnan(r["del_auc"])]
    ins_aucs = [r["ins_auc"] for r in recs if not np.isnan(r["ins_auc"])]
    rows.append({
        "tau":         tau,
        "n_probes":    len(del_aucs),
        "del_auc":     float(np.mean(del_aucs)),
        "ins_auc":     float(np.mean(ins_aucs)),
        "del_std":     float(np.std(del_aucs)),
        "ins_std":     float(np.std(ins_aucs)),
    })

sens_df = pd.DataFrame(rows).set_index("tau")

# Append RISE baseline row
if os.path.exists(RISE_EVAL_CSV):
    rise_eval = pd.read_csv(RISE_EVAL_CSV)
    rise_eval = rise_eval[rise_eval["probe_path"].isin(sample_paths)]
    if len(rise_eval):
        rise_row = {
            "n_probes": len(rise_eval),
            "del_auc":  rise_eval["del_auc_margin"].mean(),
            "ins_auc":  rise_eval["ins_auc_margin"].mean(),
            "del_std":  rise_eval["del_auc_margin"].std(),
            "ins_std":  rise_eval["ins_auc_margin"].std(),
        }
        rise_df = pd.DataFrame([rise_row], index=pd.Index(["RISE (baseline)"], name="tau"))
        display_df = pd.concat([rise_df, sens_df])
    else:
        display_df = sens_df
else:
    display_df = sens_df

print("=" * 65)
print(" TAU SENSITIVITY TABLE")
print("=" * 65)
print(f"{'Method / tau':>18}  {'n':>5}  {'Del AUC':>10}  {'Ins AUC':>10}")
print("-" * 65)
for idx_val, row in display_df.iterrows():
    marker = " ←" if str(idx_val) == "0.1" else ""
    print(f"{str(idx_val):>18}  {int(row['n_probes']):>5}  "
          f"{row['del_auc']:>10.4f}  {row['ins_auc']:>10.4f}{marker}")
print("=" * 65)
print("(Lower deletion = better  |  Higher insertion = better)")

sens_df.to_csv(os.path.join(TAU_ABL_DIR, "tau_sensitivity_table.csv"))
print(f"\nSaved: {TAU_ABL_DIR}/tau_sensitivity_table.csv")

In [ ]:
# ---------------------------------------------------------------------------
# Figure: Deletion + Insertion AUC vs tau (log-scale x-axis)
# RISE baseline shown as horizontal dashed lines for reference.
# ---------------------------------------------------------------------------

taus_plot  = [t for t in TAU_VALUES if t in sens_df.index]
del_vals   = [sens_df.loc[t, "del_auc"] for t in taus_plot]
ins_vals   = [sens_df.loc[t, "ins_auc"] for t in taus_plot]
del_stds   = [sens_df.loc[t, "del_std"] / np.sqrt(sens_df.loc[t, "n_probes"])
              for t in taus_plot]
ins_stds   = [sens_df.loc[t, "ins_std"] / np.sqrt(sens_df.loc[t, "n_probes"])
              for t in taus_plot]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
x = np.arange(len(taus_plot))
tau_labels = [str(t) for t in taus_plot]

# ── Deletion AUC ──────────────────────────────────────────────────────────
ax1.plot(x, del_vals, "o-", color="#4E79A7", lw=2.5, ms=8, label="CRISE")
ax1.fill_between(x, [v - e for v, e in zip(del_vals, del_stds)],
                    [v + e for v, e in zip(del_vals, del_stds)],
                 color="#4E79A7", alpha=0.15)
if "RISE (baseline)" in display_df.index:
    rise_del = display_df.loc["RISE (baseline)", "del_auc"]
    ax1.axhline(rise_del, color="#E15759", ls="--", lw=1.8,
                label=f"RISE baseline ({rise_del:.4f})")
ax1.set_xticks(x); ax1.set_xticklabels(tau_labels, fontsize=11)
ax1.set_xlabel("Softmax temperature τ", fontsize=10)
ax1.set_ylabel("Mean Deletion AUC (↓ better)", fontsize=10)
ax1.set_title("Deletion AUC vs τ", fontsize=11)
ax1.legend(fontsize=9); ax1.grid(alpha=0.35)

# ── Insertion AUC ─────────────────────────────────────────────────────────
ax2.plot(x, ins_vals, "o-", color="#59A14F", lw=2.5, ms=8, label="CRISE")
ax2.fill_between(x, [v - e for v, e in zip(ins_vals, ins_stds)],
                    [v + e for v, e in zip(ins_vals, ins_stds)],
                 color="#59A14F", alpha=0.15)
if "RISE (baseline)" in display_df.index:
    rise_ins = display_df.loc["RISE (baseline)", "ins_auc"]
    ax2.axhline(rise_ins, color="#E15759", ls="--", lw=1.8,
                label=f"RISE baseline ({rise_ins:.4f})")
ax2.set_xticks(x); ax2.set_xticklabels(tau_labels, fontsize=11)
ax2.set_xlabel("Softmax temperature τ", fontsize=10)
ax2.set_ylabel("Mean Insertion AUC (↑ better)", fontsize=10)
ax2.set_title("Insertion AUC vs τ", fontsize=11)
ax2.legend(fontsize=9); ax2.grid(alpha=0.35)

# Annotate the default tau
for ax, vals in [(ax1, del_vals), (ax2, ins_vals)]:
    if TAU_DEFAULT in taus_plot:
        xi = taus_plot.index(TAU_DEFAULT)
        ax.annotate("default τ",
                    xy=(xi, vals[xi]),
                    xytext=(xi + 0.3, vals[xi]),
                    fontsize=8, color="#4E79A7",
                    arrowprops=dict(arrowstyle="->", color="#4E79A7", lw=1))

plt.suptitle(f"CRISE-ID τ sensitivity  (n={len(taus_plot)} values, "
             f"N_SAMPLE={N_SAMPLE}, EVAL_STEPS={EVAL_STEPS})",
             fontsize=12)
plt.tight_layout()
fig_path = os.path.join(TAU_ABL_DIR, "fig_tau_sensitivity.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Paired comparison: for each probe present in ALL tau runs, compare
# del/ins AUC across tau values.
# Confirms whether the τ=0.1 advantage is consistent across probes or
# driven by a small subset.
# ---------------------------------------------------------------------------

# Build probe_path -> {tau -> {del, ins}} mapping
tau01_map = {
    row["probe_path"]: {"del_auc": row["del_auc_margin"], "ins_auc": row["ins_auc_margin"]}
    for _, row in tau01_df[tau01_df["probe_path"].isin(sample_paths)].iterrows()
}

# Collect per-probe records
all_records = []
for probe in sample:
    ppath = probe["img_path"]
    row = {"probe_path": ppath}
    ok = True
    for tau in TAU_VALUES:
        if tau == TAU_DEFAULT:
            if ppath not in tau01_map:
                ok = False; break
            row[f"del_{tau}"] = tau01_map[ppath]["del_auc"]
            row[f"ins_{tau}"] = tau01_map[ppath]["ins_auc"]
        else:
            found = [r for r in auc_results.get(tau, [])
                     if not np.isnan(r["del_auc"])]
            # We don't track per-probe for other taus yet; skip paired for now
            ok = False; break
    if ok:
        all_records.append(row)

if all_records:
    paired_df = pd.DataFrame(all_records)
    print(f"Paired probes (present in all tau runs): {len(paired_df)}")
    for tau in TAU_VALUES:
        if f"del_{tau}" in paired_df.columns:
            print(f"  tau={tau}  del={paired_df[f'del_{tau}'].mean():.4f}  "
                  f"ins={paired_df[f'ins_{tau}'].mean():.4f}")
else:
    print("Per-probe pairing across all tau values requires running the ablation first.")
    print("After the full run, re-run this cell to get paired statistics.")

In [ ]:
# ---------------------------------------------------------------------------
# Visual inspection: for a single probe, plot the saliency map at each
# tau value side by side.  Shows how sharpness changes with temperature.
# ---------------------------------------------------------------------------

# Pick the first probe from the sample that has maps in all tau dirs
probe_for_vis = None
for probe in sample:
    probe_path = probe["img_path"]
    true_id    = probe["true_id"]
    img_stem   = os.path.splitext(os.path.basename(probe_path))[0]

    maps_found = {}

    # τ=0.1: look in CRISE_MAP_DIR
    for f in os.listdir(CRISE_MAP_DIR):
        if (f.startswith(f"crise_tau0p1_{true_id}_{img_stem}_")
                and f.endswith("_saliency_norm.npy")):
            maps_found[0.1] = os.path.join(CRISE_MAP_DIR, f)
            break

    # Other tau values
    for tau in TAU_VALUES:
        if tau == TAU_DEFAULT:
            continue
        tau_str = f"{tau:.3g}".replace(".", "p")
        tau_dir = os.path.join(TAU_ABL_DIR, f"tau_{tau_str}")
        if not os.path.isdir(tau_dir):
            continue
        for f in os.listdir(tau_dir):
            if (f.startswith(f"crise_tau{tau_str}_{true_id}_{img_stem}_")
                    and f.endswith("_saliency_norm.npy")):
                maps_found[tau] = os.path.join(tau_dir, f)
                break

    if len(maps_found) == len(TAU_VALUES):
        probe_for_vis = probe
        maps_for_vis  = maps_found
        break

if probe_for_vis is None:
    print("No probe found with maps at all tau values — run CRISE for other tau values first.")
else:
    img = __import__("cv2").imread(probe_for_vis["img_path"])
    chip = build_aligned_chip_112(img, app)
    chip_rgb = __import__("cv2").cvtColor(chip, __import__("cv2").COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, len(TAU_VALUES) + 1,
                             figsize=(3.2 * (len(TAU_VALUES) + 1), 3.8))
    axes[0].imshow(chip_rgb)
    axes[0].set_title("Aligned chip", fontsize=9)
    axes[0].axis("off")

    for j, tau in enumerate(TAU_VALUES):
        sal = np.load(maps_for_vis[tau]).astype(np.float32)
        axes[j + 1].imshow(sal, cmap="inferno", vmin=0, vmax=1)
        marker = " ★" if tau == TAU_DEFAULT else ""
        axes[j + 1].set_title(f"τ = {tau}{marker}", fontsize=10)
        axes[j + 1].axis("off")

    plt.suptitle(
        f"Saliency map sensitivity to τ — {probe_for_vis['true_id']}",
        fontsize=11)
    plt.tight_layout()
    vis_path = os.path.join(TAU_ABL_DIR, "fig_tau_visual_comparison.png")
    plt.savefig(vis_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {vis_path}")

---
## N Convergence Analysis

τ is the only CRISE-specific hyperparameter. N (mask count) is inherited from RISE and
needs a convergence check, not a full ablation.

**Question:** Do maps converge at N=1000? We run CRISE at N ∈ {200, 500, 1000, 2000} on
a small fixed sample, then plot cosine similarity vs. N=1000 (the reference).

**Expected result:** `cos(N=1000, N=2000) > 0.97` — confirming N=1000 is sufficient
and doubling to 2000 adds negligible improvement.

In [ ]:
# ---------------------------------------------------------------------------
# N convergence: run CRISE at N in {200, 500, 2000} on a small fixed sample.
# N=1000 maps are already cached at TAU_DEFAULT — used as the reference.
# ---------------------------------------------------------------------------

N_CONV_SAMPLE = 30       # small enough to be fast
N_CONV_SEED   = 7        # different from main sample seed
N_VALUES      = [200, 500, 2000]   # 1000 already cached
CONV_DIR      = os.path.join(TAU_ABL_DIR, "n_convergence")
os.makedirs(CONV_DIR, exist_ok=True)

# Fixed convergence sample (different from tau ablation sample to avoid overlap)
rng_conv   = np.random.default_rng(N_CONV_SEED)
idx_conv   = rng_conv.choice(len(valid_probes),
                              size=min(N_CONV_SAMPLE, len(valid_probes)),
                              replace=False)
conv_sample = [valid_probes[i] for i in idx_conv]
print(f"Convergence sample: {len(conv_sample)} probes")

# Run CRISE at each N value (idempotent)
for N_val in N_VALUES:
    n_str  = str(N_val)
    n_dir  = os.path.join(CONV_DIR, f"N{N_val}")
    os.makedirs(n_dir, exist_ok=True)
    n_done = 0; n_skip = 0

    for i, probe in enumerate(tqdm(conv_sample, desc=f"CRISE N={N_val}")):
        probe_path = probe["img_path"]
        true_id    = probe["true_id"]
        img_stem   = os.path.splitext(os.path.basename(probe_path))[0]
        run_seed   = MASTER_SEED * 10_000 + i * 100   # same seed as main run

        tau_str  = f"{TAU_DEFAULT:.3g}".replace(".", "p")
        prefix   = f"crise_tau{tau_str}"
        expected = os.path.join(
            n_dir,
            f"{prefix}_{true_id}_{img_stem}_N{N_val}_s{S}_p{P1}_seed{run_seed}_saliency_norm.npy"
        )
        if os.path.exists(expected):
            n_skip += 1
            continue
        try:
            run_crise(
                true_id    = true_id,
                probe_path = probe_path,
                G=G, id_to_index=id_to_index, rec=rec, app=app,
                run_seed   = run_seed,
                out_dir    = n_dir,
                tau        = TAU_DEFAULT,
                N          = N_val,
                s          = S,
                p1         = P1,
            )
            n_done += 1
        except Exception as e:
            pass

    print(f"  N={N_val}: done={n_done}  skipped={n_skip}")


In [ ]:
# ---------------------------------------------------------------------------
# Convergence figure: cosine similarity vs N, using N=1000 as reference.
# For each probe in the convergence sample, compute cos(map_N, map_1000).
# ---------------------------------------------------------------------------

import cv2 as _cv2

# Locate N=1000 reference maps (from CRISE_MAP_DIR, tau=0.1)
tau_str = f"{TAU_DEFAULT:.3g}".replace(".", "p")
prefix  = f"crise_tau{tau_str}"

ref_maps = {}   # probe_path -> (sal_1000, chip_bgr)
for probe in conv_sample:
    probe_path = probe["img_path"]
    true_id    = probe["true_id"]
    img_stem   = os.path.splitext(os.path.basename(probe_path))[0]

    candidates = [
        f for f in os.listdir(CRISE_MAP_DIR)
        if f.startswith(f"{prefix}_{true_id}_{img_stem}_N{N_MASKS}_")
        and f.endswith("_saliency_norm.npy")
        and not os.path.dirname(f)   # root dir only, not subdirs
    ]
    if not candidates:
        continue
    sal_path = os.path.join(CRISE_MAP_DIR, candidates[0])
    ref_maps[probe_path] = np.load(sal_path).ravel().astype(np.float32)

print(f"Reference maps found (N=1000): {len(ref_maps)} / {len(conv_sample)}")

# Compute per-probe cosine similarity at each N value
all_N    = sorted(N_VALUES) + [N_MASKS]   # e.g. [200, 500, 1000, 2000]
cos_by_N = {N: [] for N in all_N}
cos_by_N[N_MASKS] = [1.0] * len(ref_maps)   # reference vs itself = 1.0

for N_val in N_VALUES:
    n_dir  = os.path.join(CONV_DIR, f"N{N_val}")
    if not os.path.isdir(n_dir):
        continue

    for probe in conv_sample:
        probe_path = probe["img_path"]
        true_id    = probe["true_id"]
        img_stem   = os.path.splitext(os.path.basename(probe_path))[0]

        if probe_path not in ref_maps:
            continue

        candidates = [
            f for f in os.listdir(n_dir)
            if f.startswith(f"{prefix}_{true_id}_{img_stem}_N{N_val}_")
            and f.endswith("_saliency_norm.npy")
        ]
        if not candidates:
            continue

        sal = np.load(os.path.join(n_dir, candidates[0])).ravel().astype(np.float32)
        ref = ref_maps[probe_path]
        ref_n = ref / (np.linalg.norm(ref) + 1e-9)
        sal_n = sal / (np.linalg.norm(sal) + 1e-9)
        cos_by_N[N_val].append(float(ref_n @ sal_n))

# Summary stats
print("\nCosine similarity vs N=1000 reference:")
print(f"  {'N':>6}  {'mean':>8}  {'std':>8}  {'min':>8}  n")
for N_val in sorted(all_N):
    vals = cos_by_N[N_val]
    if not vals:
        continue
    print(f"  {N_val:>6}  {np.mean(vals):>8.4f}  {np.std(vals):>8.4f}"
          f"  {np.min(vals):>8.4f}  {len(vals)}")

# ── Figure ────────────────────────────────────────────────────────────────
plot_N    = sorted([N for N in all_N if cos_by_N[N]])
mean_vals = [np.mean(cos_by_N[N]) for N in plot_N]
std_vals  = [np.std(cos_by_N[N])  for N in plot_N]

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(range(len(plot_N)), mean_vals, "o-",
        color="#4E79A7", lw=2.5, ms=8, zorder=5)
ax.fill_between(range(len(plot_N)),
                [m - s for m, s in zip(mean_vals, std_vals)],
                [m + s for m, s in zip(mean_vals, std_vals)],
                color="#4E79A7", alpha=0.15, label="mean ± 1 std")

# Mark N=1000 reference
ref_idx = plot_N.index(N_MASKS)
ax.axvline(ref_idx, color="#E15759", ls="--", lw=1.5,
           label=f"N={N_MASKS} (reference, τ=0.1 default)")
ax.axhline(0.97, color="gray", ls=":", lw=1, label="0.97 convergence line")

ax.set_xticks(range(len(plot_N)))
ax.set_xticklabels([str(N) for N in plot_N], fontsize=11)
ax.set_xlabel("Number of masks N", fontsize=11)
ax.set_ylabel("Cosine similarity vs N=1000 map", fontsize=10)
ax.set_title("N convergence: saliency maps stabilise at N=1000", fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
conv_fig = os.path.join(CONV_DIR, "fig_N_convergence.png")
plt.savefig(conv_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {conv_fig}")

# Conclusion
if cos_by_N.get(2000):
    cos_2000 = np.mean(cos_by_N[2000])
    cos_500  = np.mean(cos_by_N.get(500, [0.0]))
    print("\n" + "=" * 50)
    print(" N CONVERGENCE CONCLUSION")
    print("=" * 50)
    if cos_2000 >= 0.97:
        print(f"  cos(N=1000, N=2000) = {cos_2000:.4f} >= 0.97")
        print("  CONFIRMED: N=1000 is sufficient. Doubling to N=2000")
        print("  adds negligible improvement.")
    else:
        print(f"  cos(N=1000, N=2000) = {cos_2000:.4f} < 0.97")
        print("  Consider increasing N to 2000 for final results.")
    print(f"  cos(N=500,  N=1000) = {cos_500:.4f}")
    print("=" * 50)


In [ ]:
# ---------------------------------------------------------------------------
# Summary — confirm or update the recommended default tau
# ---------------------------------------------------------------------------

if sens_df.empty:
    print("No results yet — run the cells above.")
else:
    best_del = sens_df["del_auc"].idxmin()
    best_ins = sens_df["ins_auc"].idxmax()

    print("=" * 55)
    print(" TAU ABLATION SUMMARY")
    print("=" * 55)
    print(f"  Best deletion AUC  : tau = {best_del}"
          f"  ({sens_df.loc[best_del, 'del_auc']:.4f})")
    print(f"  Best insertion AUC : tau = {best_ins}"
          f"  ({sens_df.loc[best_ins, 'ins_auc']:.4f})")
    print()

    if best_del == best_ins == TAU_DEFAULT:
        print(f"  CONCLUSION: tau={TAU_DEFAULT} is optimal on both metrics.")
        print("  The default in crise.py does not need to change.")
    elif best_del == best_ins:
        print(f"  CONCLUSION: tau={best_del} is optimal on both metrics.")
        print(f"  Consider updating TAU in crise.py from {TAU_DEFAULT} to {best_del}.")
    else:
        print(f"  NOTE: Best deletion tau ({best_del}) != best insertion tau ({best_ins}).")
        print(f"  Recommend tau={TAU_DEFAULT} as default (balanced trade-off).")
        print("  Report full table in the paper sensitivity section.")
    print("=" * 55)